# RAG Pipeline Demo
## Retrieval-Augmented Generation with DeepSeek-R1

This notebook demonstrates a RAG pipeline that grounds an LLM with content 
directly retrieved from a research paper — **DeepSeek-R1** (Guo et al., 2025).

**Stack:** Python · OpenAI API · ChromaDB · Docling · LangChain Text Splitters

**The core idea:** Even when an LLM has seen a paper during training, specific 
details (e.g. exact benchmark scores, algorithm names, author arguments) are often 
recalled imprecisely or fabricated. RAG injects the exact source text as context, 
reducing hallucination on document-specific questions.

📄 Paper: [DeepSeek-R1 on arXiv](https://arxiv.org/abs/2501.12948)

In [31]:
# Imports
import requests
from pathlib import Path
from openai import OpenAI
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv
import os


In [ ]:
# Configs
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
DOCLING_URL = os.getenv("DOCLING_URL", "http://localhost:5001")
CHUNK_SIZE = 512
CHUNK_OVERLAP = 50
PAPER_URL = "https://arxiv.org/pdf/2501.12948"

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print("Config loaded!")

Config loaded!


In [22]:
# Download Paper
Path("data/raw").mkdir(parents=True, exist_ok=True)

pdf_path = "data/raw/deepseek_r1.pdf"

if not Path(pdf_path).exists():
    print("Downloading paper...")
    response = requests.get(PAPER_URL)
    with open(pdf_path, "wb") as f:
        f.write(response.content)
    print("Done!")
else:
    print("Paper already downloaded.")

Paper already downloaded.


In [23]:
# Extract text via Docling
Path("data/processed").mkdir(parents=True, exist_ok=True)

md_path = "data/processed/deepseek_r1.md"

if not Path(md_path).exists():
    print("Extracting text via Docling...")
    with open(pdf_path, "rb") as f:
        response = requests.post(
            f"{DOCLING_URL}/v1/convert/file",
            files={"files": ("deepseek_r1.pdf", f, "application/pdf")},
            data={"to_formats": "md"}
        )
    
    result = response.json()
    markdown = result["document"]["md_content"]
    
    with open(md_path, "w", encoding="utf-8") as f:
        f.write(markdown)
    print(f"Done! Extracted {len(markdown)} characters.")
else:
    print("Already extracted.")

Already extracted.


In [24]:
# Chunk text and load into ChromaDB
print("Loading extracted text...")
with open(md_path, "r", encoding="utf-8") as f:
    text = f.read()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)
chunks = splitter.split_text(text)
print(f"Created {len(chunks)} chunks")

# Setup ChromaDB
client = chromadb.Client()
collection = client.get_or_create_collection("deepseek_r1")

# Setup Embeddings
openai_client = OpenAI(api_key=OPENAI_API_KEY)

print("Creating embeddings and loading into ChromaDB...")
batch_size = 100
for i in range(0, len(chunks), batch_size):
    batch = chunks[i:i+batch_size]
    response = openai_client.embeddings.create(
        input=batch,
        model="text-embedding-3-small"
    )
    embeddings = [r.embedding for r in response.data]
    collection.add(
        documents=batch,
        embeddings=embeddings,
        ids=[f"chunk_{j}" for j in range(i, i+len(batch))]
    )
    print(f"  {min(i+batch_size, len(chunks))}/{len(chunks)} chunks loaded")

print("Done!")

Loading extracted text...
Created 8574 chunks
Creating embeddings and loading into ChromaDB...
  100/8574 chunks loaded
  200/8574 chunks loaded
  300/8574 chunks loaded
  400/8574 chunks loaded
  500/8574 chunks loaded
  600/8574 chunks loaded
  700/8574 chunks loaded
  800/8574 chunks loaded
  900/8574 chunks loaded
  1000/8574 chunks loaded
  1100/8574 chunks loaded
  1200/8574 chunks loaded
  1300/8574 chunks loaded
  1400/8574 chunks loaded
  1500/8574 chunks loaded
  1600/8574 chunks loaded
  1700/8574 chunks loaded
  1800/8574 chunks loaded
  1900/8574 chunks loaded
  2000/8574 chunks loaded
  2100/8574 chunks loaded
  2200/8574 chunks loaded
  2300/8574 chunks loaded
  2400/8574 chunks loaded
  2500/8574 chunks loaded
  2600/8574 chunks loaded
  2700/8574 chunks loaded
  2800/8574 chunks loaded
  2900/8574 chunks loaded
  3000/8574 chunks loaded
  3100/8574 chunks loaded
  3200/8574 chunks loaded
  3300/8574 chunks loaded
  3400/8574 chunks loaded
  3500/8574 chunks loaded
  36

In [33]:
SYSTEM_PROMPT = "You are a helpful research assistant. Be precise and cite specific details where possible."

def query_without_rag(question: str) -> str:
    completion = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question}
        ]
    )
    return completion.choices[0].message.content

def query_paper(question: str, n_results: int = 10) -> str:
    response = openai_client.embeddings.create(
        input=[question],
        model="text-embedding-3-small"
    )
    query_embedding = response.data[0].embedding
    
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results
    )
    context = "\n\n".join(results["documents"][0])
    
    completion = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Context from the paper:\n\n{context}\n\nQuestion: {question}"}
        ]
    )
    return completion.choices[0].message.content

In [34]:
# Demo Query 1: Training methodology
question = "What exact AIME 2024 benchmark score did DeepSeek-R1 achieve and how does it compare to OpenAI o1-1217?"


print("=== WITHOUT RAG ===")
print(query_without_rag(question))
print("\n=== WITH RAG ===")
print(query_paper(question))

=== WITHOUT RAG ===
As of my last update in October 2023, I do not have specific information regarding the AIME 2024 benchmark scores for DeepSeek-R1 or OpenAI o1-1217. For the most accurate and timely benchmarks or scores, I recommend checking official sources such as the organizations' reports or relevant research publications that document these specific results.

=== WITH RAG ===
DeepSeek-R1 achieved a score of 11.3 out of 15 on the AIME 2025 benchmark. In comparison, OpenAI o1-1217 scored 12.0 out of 15 on the same benchmark. This indicates that DeepSeek-R1's performance is very close to that of OpenAI o1-1217, with a difference of only 0.7 points.


In [35]:
question = "What are the limitations of DeepSeek-R1 mentioned by the authors?"

print("=== WITHOUT RAG ===")
print(query_without_rag(question))
print("\n=== WITH RAG ===")
print(query_paper(question))

=== WITHOUT RAG ===
DeepSeek-R1, as discussed by its authors, has several limitations that need to be acknowledged. Here are the main points:

1. **Data Dependency**: DeepSeek-R1's effectiveness heavily relies on the availability of high-quality training data. If the dataset is limited or biased, it can affect the model's performance and generalization.

2. **Computational Requirements**: The model's complex architecture requires significant computational resources for training and inference, which may limit its accessibility for researchers or practitioners with limited infrastructure.

3. **Interpretability**: Like many deep learning models, DeepSeek-R1 may suffer from a lack of interpretability, making it challenging to understand the decision-making process behind predictions.

4. **Generalization Challenges**: While DeepSeek-R1 has shown promising results on specific datasets, there is a concern about its ability to generalize across different types of data or real-world scenarios

In [36]:
question = "What is GRPO and why did DeepSeek use it instead of PPO?"

print("=== WITHOUT RAG ===")
print(query_without_rag(question))
print("\n=== WITH RAG ===")
print(query_paper(question))

=== WITHOUT RAG ===
GRPO, or Generalized Reinforcement Policy Optimization, is an algorithm used in reinforcement learning that extends the capabilities of traditional policy optimization methods like Proximal Policy Optimization (PPO). GRPO is designed to allow for more flexible exploration and adaptation in complex environments, enhancing the efficiency and performance of learning agents.

DeepSeek, a notable research initiative or organization, likely opted for GRPO over PPO for its potential advantages in certain scenarios. While PPO is known for its simplicity and effectiveness in many standard reinforcement learning tasks, GRPO may offer improved convergence properties or better performance in high-dimensional action spaces or environments with a more complex reward structure. This can be particularly advantageous in environments where PPO may struggle, such as those with sparse rewards or requiring sophisticated exploration strategies.

The choice of GRPO over PPO would typicall

## Summary

We built a minimal RAG pipeline to extract and query knowledge from a PDF paper.

**Workflow:** PDF → Markdown (Docling) → Chunks → Embeddings (OpenAI) → Vector Store (ChromaDB) → LLM Query (GPT-4o-mini)


**Stack:** Python · OpenAI API · ChromaDB · Docling · LangChain Text Splitters  
**Paper:** [DeepSeek-R1 (Guo et al., 2025)](https://arxiv.org/abs/2501.12948)

| Query | Without RAG | With RAG |
|-------|-------------|----------|
| AIME 2024 score | No specific knowledge | 11.3/15 on AIME 2025, vs OpenAI o1-1217 with 12.0/15 |
| GRPO vs PPO | Hallucinated "Generalized Reinforcement Policy Optimization" | Correct: Group Relative Policy Optimization (Shao et al. 2024) |
| Limitations | Generic / fabricated limitations | Readability issues, language mixing, reasoning degradation |

This demo shows that even a simple RAG pipeline produces more grounded, 
document-specific outputs compared to relying on model memory alone.